# Milestone 1.3: Causal Gold Technical Features & Entity News Collector

## 1. What We Built
- **`GoldFeatureBuilder` (`src/stockcortex/features/gold_builder.py`)**: Engineered dimensionless momentum (`log_ret`, `price_to_sma`, normalized MACD, ROC), microstructure/volatility (`Wilder RSI`, `Bollinger %B`, `ATR/Close`, `Garman-Klass 21d vol`, `Volume Z-Score`), and cross-sectional regime signals (`63d Beta` and `21d Relative Strength` vs. NIFTY 50).
- **`EntityNewsLoader` (`src/stockcortex/ingestion/news_loader.py`)**: Built a dual-provider Indian financial news collector (NewsData.io + Google News India RSS fallback) with entity keyword filtering and post-15:30 IST timestamp shifting.

## 2. Why These Architectural Decisions Were Made
| Engineering Choice | Leakage / Quantitative Safeguard |
| :--- | :--- |
| **Dimensionless Ratios & Log Returns** | Raw rupee prices (e.g., ₹150 vs. ₹4,500) are non-stationary across stocks and eras; ratios (`Close/SMA - 1`, `ATR/Close`) normalize scale cleanly for tree models. |
| **Garman-Klass Range Volatility** | Uses Open/High/Low/Close range information rather than close-to-close returns alone, capturing intraday tail risk with lower estimator variance. |
| **Post-15:30 IST News Shifting (`t -> t+1`)** | NSE continuous trading closes at `15:30 IST`. Any headline published at `15:31 IST` or later cannot be acted upon before day `t`'s close; shifting it to `t+1` eliminates look-ahead leakage. |

## 3. How to Build Gold Features & Benchmark DuckDB Query Latency

In [1]:
import time
from stockcortex.ingestion.market_loader import MarketDataLoader
from stockcortex.ingestion.news_loader import EntityNewsLoader
from stockcortex.features.gold_builder import GoldFeatureBuilder
from stockcortex.storage.duck_store import DuckStore
from stockcortex.config import settings

# 1. Ingest NIFTY 50 (^NSEI) benchmark & build Gold Parquet features for starter universe
MarketDataLoader().ingest_nifty50_benchmark(period="5y")
gold_paths = GoldFeatureBuilder().build_universe()

# 2. Fetch & align entity-filtered news for RELIANCE and TCS
news_loader = EntityNewsLoader()
for sym in ["RELIANCE", "TCS"]:
    news_loader.ingest_symbol_news(sym)

# 3. Verify <1s DuckDB query latency across Gold Parquet
t0 = time.perf_counter()
gold_glob = (settings.gold_dir / "*_features.parquet").as_posix()
df_check = DuckStore().query_parquet(f"""
    SELECT symbol, COUNT(*) AS rows, MAX(date) AS latest_date,
           ROUND(AVG(rsi_14), 2) AS avg_rsi, ROUND(AVG(beta_nifty_63d), 3) AS avg_beta
    FROM read_parquet('{gold_glob}')
    GROUP BY symbol ORDER BY symbol
""")
elapsed_ms = (time.perf_counter() - t0) * 1000
print(f"DuckDB Gold Scan Latency: {elapsed_ms:.2f} ms")
df_check

DuckDB Gold Scan Latency: 125.45 ms


,symbol,rows,latest_date,avg_rsi,avg_beta
0,BHARTIARTL,1236,2026-11-26,54.40,0.782
1,HDFCBANK,1236,2026-11-26,51.07,1.072
2,HINDUNILVR,1236,2026-11-26,49.29,0.579
3,ICICIBANK,1236,2026-11-26,53.27,0.931
4,INFY,1236,2026-11-26,49.84,1.069
5,ITC,1236,2026-11-26,51.72,0.640
6,LT,1236,2026-11-26,54.00,1.117
7,RELIANCE,1236,2026-11-26,50.94,1.150
8,SBIN,1236,2026-11-26,53.63,1.110
9,TCS,1236,2026-11-26,49.60,0.890
